# 1.2 Variant class, target-gene, and gene-percentage bargraphs — Run2

This notebook reads the Run2 per-variant QC file directly from the cluster and recreates the plots from the earlier **1.2 variant/gene-specific bargraph** notebook.

It generates:

- combined mean variant-class counts for Run2;
- combined mean variant counts across the 16 target genes;
- caller-specific variant-class plots comparing DeepVariant and Mutect2;
- mean percentage of each sample's total retained variants assigned to each of the 16 target genes.

The percentage calculation uses **all retained variants in each sample as the denominator**, not only variants in the 16 target genes. Therefore, the 16 gene percentages are not expected to sum to 100%.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

run_label = "Run2"

input_csv = Path(
    "/home/donetski/Notebooks/OutputFiles/04_qc_checking_on_target/"
    "04_Run2_per_variant_target_status_FULL.csv"
)

output_dir = (
    Path("/home/donetski/Notebooks/OutputFiles/")
    / "NEW_1.2_variant_gene_specific_bargraphs"
    / run_label.lower()
)

figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

output_prefix = "1.2"

filter_to_pass = True
filter_to_on_target = True
combine_mode = "average"  # options: "average" or "sum"

callers = ["DeepVariant", "Mutect2"]
variant_classes = ["SNV", "insertion", "deletion", "indel", "substitution"]

TARGET_GENES = [
    "ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53",
]

tick_fontsize = 15
axis_label_fontsize = 18
title_fontsize = 20
legend_fontsize = 12

axis_labelpad = 8
title_pad = 10

show_plots = True

print("Input file:", input_csv)
print("Output folder:", output_dir)


## Load the Run2 per-variant QC table

By default, this section retains only:

- rows with `FILTER == "PASS"`;
- rows marked as on-target.

Caller names and gene symbols are standardized before the summaries are calculated.


In [ ]:
def read_csv_auto(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin1")


def require_columns(input_df, columns):
    missing = [column for column in columns if column not in input_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


df = read_csv_auto(input_csv)
require_columns(df, ["Sample.ID", "caller", "Gene", "Variant.Class"])

df_qc = df.copy()

if filter_to_pass:
    require_columns(df_qc, ["FILTER"])
    df_qc = df_qc[df_qc["FILTER"].astype(str).str.upper().eq("PASS")].copy()

if filter_to_on_target:
    require_columns(df_qc, ["on_target"])
    on_target_mask = df_qc["on_target"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_qc = df_qc[on_target_mask].copy()

caller_lookup = {caller.lower(): caller for caller in callers}
df_qc["caller"] = df_qc["caller"].astype(str).str.strip().str.lower().map(caller_lookup)
df_qc["Gene"] = df_qc["Gene"].astype(str).str.strip().str.upper()
df_qc["Variant.Class"] = df_qc["Variant.Class"].astype(str).str.strip()
df_qc = df_qc[df_qc["caller"].isin(callers)].copy()

print("Input rows:", f"{len(df):,}")
print("Rows after selected QC filters:", f"{len(df_qc):,}")
print("Unique samples:", f"{df_qc['Sample.ID'].nunique():,}")
print()
print(df_qc.groupby("caller")["Sample.ID"].nunique().rename("unique_samples"))


## Calculate mean counts by caller

For each caller, variant counts are first calculated separately for every sample.

Samples with no variants in a particular class or gene are assigned a count of zero before the mean is calculated. The DeepVariant and Mutect2 means are then combined using `combine_mode`.


In [ ]:
def mean_count_summary(input_df, category_col, categories):
    summary = pd.DataFrame(index=categories)

    for caller in callers:
        caller_df = input_df[input_df["caller"].eq(caller)]
        sample_ids = caller_df["Sample.ID"].dropna().unique()

        counts = pd.crosstab(caller_df["Sample.ID"], caller_df[category_col])
        counts = counts.reindex(index=sample_ids, columns=categories, fill_value=0)
        summary[caller] = counts.mean(axis=0) if len(sample_ids) else 0

    if combine_mode == "average":
        summary["combined_mean"] = summary[callers].mean(axis=1)
    elif combine_mode == "sum":
        summary["combined_mean"] = summary[callers].sum(axis=1)
    else:
        raise ValueError("combine_mode must be 'average' or 'sum'")

    return summary.rename_axis(category_col).reset_index()


variant_class_summary = mean_count_summary(df_qc, "Variant.Class", variant_classes)
target_gene_summary = mean_count_summary(df_qc, "Gene", TARGET_GENES)

variant_class_summary.to_csv(
    output_dir / f"{output_prefix}_{run_label}_variant_class_mean_values.csv",
    index=False,
)
target_gene_summary.to_csv(
    output_dir / f"{output_prefix}_{run_label}_target_gene_mean_values.csv",
    index=False,
)

variant_class_summary


## Variant-class counts — Run2

This plot shows the combined DeepVariant/Mutect2 mean number of variants per sample for each variant class.


In [ ]:
plot_values = variant_class_summary.set_index("Variant.Class")["combined_mean"].reindex(variant_classes)

ax = plot_values.plot(kind="bar", figsize=(10, 6), label=run_label)
ax.set_xlabel("Variant class", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
ax.set_ylabel(
    "Mean variants per sample" if combine_mode == "average" else "Summed mean variants per sample",
    fontsize=axis_label_fontsize,
    labelpad=axis_labelpad,
)
ax.set_title(f"Variant class counts — {run_label}", fontsize=title_fontsize, fontweight="bold", pad=title_pad)
ax.tick_params(axis="both", labelsize=tick_fontsize)
ax.legend(fontsize=legend_fontsize, loc="upper right")
plt.xticks(rotation=45, ha="right")

out_png = figure_dir / f"{output_prefix}_{run_label}_variant_class_combined_mean.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
if show_plots:
    plt.show()
else:
    plt.close()

print("Saved:", out_png)


## Mean counts across the 16 target genes — Run2

The genes remain in the fixed 16-gene panel order used by the original 1.2 notebook.


In [ ]:
plot_values = target_gene_summary.set_index("Gene")["combined_mean"].reindex(TARGET_GENES)

ax = plot_values.plot(kind="bar", figsize=(14, 6), label=run_label)
ax.set_xlabel("Gene", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
ax.set_ylabel(
    "Mean variants per sample" if combine_mode == "average" else "Summed mean variants per sample",
    fontsize=axis_label_fontsize,
    labelpad=axis_labelpad,
)
ax.set_title(
    f"Variant counts across 16 target genes — {run_label}",
    fontsize=title_fontsize,
    fontweight="bold",
    pad=title_pad,
)
ax.tick_params(axis="both", labelsize=tick_fontsize)
ax.legend(fontsize=legend_fontsize, loc="upper right")
plt.xticks(rotation=45, ha="right")

out_png = figure_dir / f"{output_prefix}_{run_label}_target_gene_combined_mean.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
if show_plots:
    plt.show()
else:
    plt.close()

print("Saved:", out_png)


## Caller-specific variant-class plots

One figure is produced for each variant class. Each figure compares the Run2 mean from DeepVariant with the Run2 mean from Mutect2.


In [ ]:
caller_table = variant_class_summary.set_index("Variant.Class")[callers]

for variant_class in variant_classes:
    plot_values = caller_table.loc[variant_class]

    ax = plot_values.plot(kind="bar", figsize=(8, 5))
    ax.set_xlabel("Caller", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
    ax.set_ylabel("Mean variants per sample", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
    ax.set_title(
        f"{variant_class} counts by caller — {run_label}",
        fontsize=title_fontsize,
        fontweight="bold",
        pad=title_pad,
    )
    ax.tick_params(axis="both", labelsize=tick_fontsize)
    plt.xticks(rotation=0)

    safe_class = variant_class.replace(" ", "_").lower()
    out_png = figure_dir / f"{output_prefix}_{run_label}_{safe_class}_mean_by_caller.png"

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    if show_plots:
        plt.show()
    else:
        plt.close()

    print("Saved:", out_png)


## Mean percentage of total variants represented by each target gene

For every sample:

1. the denominator is the total number of retained variant rows across all genes and both callers;
2. the numerator is the number of retained variant rows assigned to a given target gene;
3. the percentage is calculated within that sample;
4. percentages are averaged across all Run2 samples.

Genes are ordered from highest to lowest Run2 mean percentage, matching the intended percentage graph.


In [ ]:
sample_ids = df_qc["Sample.ID"].dropna().unique()

sample_totals = df_qc.groupby("Sample.ID").size().reindex(sample_ids)

target_gene_counts = pd.crosstab(df_qc["Sample.ID"], df_qc["Gene"])
target_gene_counts = target_gene_counts.reindex(index=sample_ids, columns=TARGET_GENES, fill_value=0)

target_gene_percentages = target_gene_counts.div(sample_totals, axis=0).mul(100)

target_gene_percentage_summary = (
    target_gene_percentages.mean(axis=0)
    .rename("mean_pct_of_total_variants")
    .sort_values(ascending=False)
    .rename_axis("Gene")
    .reset_index()
)

target_gene_percentage_summary.to_csv(
    output_dir / f"{output_prefix}_{run_label}_target_gene_mean_percent_of_total.csv",
    index=False,
)

target_gene_percentage_summary


In [ ]:
plot_values = target_gene_percentage_summary.set_index("Gene")["mean_pct_of_total_variants"]

ax = plot_values.plot(kind="bar", figsize=(14, 6), legend=False)

ax.set_xlabel("Gene", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
ax.set_ylabel("Mean % of total variants", fontsize=axis_label_fontsize, labelpad=axis_labelpad)

ax.set_title(
    "Mean Percentage of Total Variants by Target Gene",
    fontsize=title_fontsize,
    fontweight="bold",
    pad=title_pad,
)

ax.tick_params(axis="both", labelsize=tick_fontsize)
ax.grid(axis="y", linestyle="--", linewidth=1, alpha=0.5)
ax.set_axisbelow(True)

plt.xticks(rotation=45, ha="right")


for label in ax.get_xticklabels():
    label.set_x(label.get_position()[0] + 0.4)

out_png = figure_dir / f"{output_prefix}_{run_label}_target_gene_mean_percent_of_total.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.transforms as transforms

plot_values = target_gene_percentage_summary.set_index("Gene")["mean_pct_of_total_variants"]

ax = plot_values.plot(kind="bar", figsize=(14, 6), legend=False)

ax.set_xlabel("Gene", fontsize=axis_label_fontsize, labelpad=axis_labelpad)
ax.set_ylabel("Mean % of total variants", fontsize=axis_label_fontsize, labelpad=axis_labelpad)

ax.set_title(
    "Mean Percentage of Total Variants by Target Gene",
    fontsize=title_fontsize,
    fontweight="bold",
    pad=title_pad,
)

ax.tick_params(axis="both", labelsize=tick_fontsize)
ax.grid(axis="y", linestyle="--", linewidth=1, alpha=0.5)
ax.set_axisbelow(True)

plt.xticks(rotation=45, ha="right", rotation_mode="anchor")

offset = transforms.ScaledTranslation(4/72, 0, ax.figure.dpi_scale_trans)
for label in ax.get_xticklabels():
    label.set_transform(label.get_transform() + offset)

out_png = figure_dir / f"{output_prefix}_{run_label}_target_gene_mean_percent_of_total.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

## Output check

This final cell lists the summary tables and figures written to the Run2 output folder.


In [ ]:
saved_files = sorted(path for path in output_dir.rglob("*") if path.is_file())

print(f"Saved {len(saved_files)} files:")
for path in saved_files:
    print(path)
